# Plant Guard AI ? Stage 4 PlantVillage ONNX GPU Evaluation

Evaluation only. This notebook uses the fixed 8,154-row PlantVillage test manifest and the existing production ONNX model. It does not train, fine-tune, convert, or modify weights, and it does not make a new split. Because the model is documented as PlantVillage-trained and original membership records are missing, this is not independent validation.

Every run writes into a new timestamped folder under `backend/ml/experiments/stage4_independent_validation/plantvillage_onnx_heldout_gpu/`. Stage 3 files/results and the existing PlantDoc Stage 4 baseline are not output targets.

The ONNX Runtime session provider list alone does not show which provider executed nodes. This workflow profiles a real model smoke batch and requires CUDA-assigned nodes before full evaluation. See the official [execution provider](https://onnxruntime.ai/docs/execution-providers/) and [profiling](https://onnxruntime.ai/docs/performance/tune-performance/profiling-tools.html) docs.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone

MYDRIVE = Path('/content/drive/MyDrive')
# Set this manually if the project is stored in another Drive folder.
PROJECT_OVERRIDE = None  # Example: '/content/drive/MyDrive/Projects/PlantGuard-AI'

def looks_like_project(path):
    return all((path / rel).is_file() for rel in [
        'backend/ml/models/efficientnet_v2_s_best.onnx',
        'backend/ml/models/efficientnet_v2_s_best.onnx.data',
        'backend/ml/models/classes.json',
        'backend/ml/experiments/stage3_preflight/plantvillage_test.csv',
        'datasets/plantvillage-source.zip',
        'backend/ml/predict.py',
    ])

if PROJECT_OVERRIDE:
    PROJECT = Path(PROJECT_OVERRIDE).expanduser().resolve()
    if not looks_like_project(PROJECT):
        raise FileNotFoundError(f'PROJECT_OVERRIDE is not a complete project root: {PROJECT}')
else:
    candidates = [MYDRIVE / 'PlantGuard-AI', MYDRIVE / 'PlantGuard-AI' / 'PlantGuard-AI']
    matches = [p for p in candidates if looks_like_project(p)]
    if len(matches) != 1:
        raise FileNotFoundError(
            'Could not identify exactly one Plant Guard AI project root. '
            'Expected /content/drive/MyDrive/PlantGuard-AI or one nested repository. '
            'Set PROJECT_OVERRIDE to the directory containing backend/, datasets/, and scripts/.'
        )
    PROJECT = matches[0]

MODEL_DIR = PROJECT / 'backend/ml/models'
STAGE3_DIR = PROJECT / 'backend/ml/experiments/stage3_preflight'
ARCHIVE = PROJECT / 'datasets/plantvillage-source.zip'
OUT_ROOT = PROJECT / 'backend/ml/experiments/stage4_independent_validation/plantvillage_onnx_heldout_gpu'
RUN_ID = datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%f_UTC')
RUN_DIR = OUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('Project root:', PROJECT)
print('Stage 4 run output:', RUN_DIR)


In [ ]:
%pip -q install onnxruntime-gpu pillow numpy pandas scikit-learn matplotlib


In [ ]:
import csv, hashlib, json, math, platform, sys, time, zipfile
from collections import Counter
from io import BytesIO

import numpy as np
import pandas as pd
from PIL import Image
import onnxruntime as ort
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

print('Python:', sys.version)
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__, 'CUDA build:', torch.version.cuda)
print('PyTorch CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
    print('CUDA device capability:', torch.cuda.get_device_capability(0))
print('ONNX Runtime:', ort.__version__)
print('ONNX Runtime available providers:', ort.get_available_providers())
if not torch.cuda.is_available():
    raise RuntimeError('Colab GPU is not active in PyTorch; stop before evaluation.')
if 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('CUDAExecutionProvider is unavailable in ONNX Runtime; stop before evaluation.')


In [ ]:
# Verify paths, immutable inputs, label order, and the fixed test-only manifest.
import sys
sys.path.insert(0, str(PROJECT))
from backend.ml.predict import preprocess_image, get_class_names

MODEL = MODEL_DIR / 'efficientnet_v2_s_best.onnx'
EXTERNAL_DATA = MODEL_DIR / 'efficientnet_v2_s_best.onnx.data'
CLASSES_FILE = MODEL_DIR / 'classes.json'
CLASS_NAMES_FILE = MODEL_DIR / 'class_names.txt'
MANIFEST = STAGE3_DIR / 'plantvillage_test.csv'
PRODUCTION_PREPROCESSOR = PROJECT / 'backend/ml/predict.py'

for required in [MODEL, EXTERNAL_DATA, CLASSES_FILE, CLASS_NAMES_FILE, MANIFEST, ARCHIVE, PRODUCTION_PREPROCESSOR]:
    if not required.is_file():
        raise FileNotFoundError(required)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

meta = json.loads(CLASSES_FILE.read_text(encoding='utf-8'))
classes = meta['classes']
production_classes = get_class_names()
assert meta['model_name'] == 'efficientnet_v2_s'
assert meta['image_size'] == 224 and meta['num_classes'] == 38
assert len(classes) == len(set(classes)) == 38
assert classes == production_classes, 'classes.json and backend.ml.predict.get_class_names() disagree'

rows = pd.read_csv(MANIFEST, dtype={'sha256':str})
assert len(rows) == 8154, f'Expected exactly 8154 test rows; found {len(rows)}'
assert set(rows['split']) == {'test'}, 'Manifest must contain only the fixed test split'
assert set(rows['class_name']) == set(classes), 'Test manifest does not cover all 38 classes'
assert rows['member'].is_unique, 'Duplicate ZIP member path in test manifest'
assert rows['sha256'].str.fullmatch(r'[0-9a-fA-F]{64}').all(), 'Missing or invalid per-image SHA-256'
class_counts = {str(k): int(v) for k, v in rows.groupby('class_name').size().reindex(classes, fill_value=0).to_dict().items()}
assert all(count > 0 for count in class_counts.values())

archive = zipfile.ZipFile(ARCHIVE, 'r')
archive_members = set(archive.namelist())
missing_members = sorted(set(rows['member']) - archive_members)
assert not missing_members, f'{len(missing_members)} manifest members are absent from the ZIP; first: {missing_members[:5]}'

# Hash inputs without altering them. Per-image payload hashes are rechecked as bytes are read below.
input_record = {
    'project_root': str(PROJECT),
    'model': {'path':str(MODEL.relative_to(PROJECT)), 'bytes':MODEL.stat().st_size, 'sha256':sha256_file(MODEL)},
    'external_tensor_data': {'path':str(EXTERNAL_DATA.relative_to(PROJECT)), 'bytes':EXTERNAL_DATA.stat().st_size, 'sha256':sha256_file(EXTERNAL_DATA)},
    'classes': {'path':str(CLASSES_FILE.relative_to(PROJECT)), 'bytes':CLASSES_FILE.stat().st_size, 'sha256':sha256_file(CLASSES_FILE), 'ordered_labels':classes},
    'class_names_txt': {'path':str(CLASS_NAMES_FILE.relative_to(PROJECT)), 'bytes':CLASS_NAMES_FILE.stat().st_size, 'sha256':sha256_file(CLASS_NAMES_FILE), 'matches_classes_json':True},
    'test_manifest': {'path':str(MANIFEST.relative_to(PROJECT)), 'bytes':MANIFEST.stat().st_size, 'sha256':sha256_file(MANIFEST), 'rows':len(rows), 'split':'test', 'class_counts':class_counts},
    'dataset_archive': {'path':str(ARCHIVE.relative_to(PROJECT)), 'bytes':ARCHIVE.stat().st_size, 'zip_member_count':len(archive_members), 'full_archive_sha256':'not computed; each of the 8154 selected image payloads is hashed and checked during reading'},
    'production_preprocessor': {'path':str(PRODUCTION_PREPROCESSOR.relative_to(PROJECT)), 'sha256':sha256_file(PRODUCTION_PREPROCESSOR)},
}
(RUN_DIR / 'input_hashes.json').write_text(json.dumps(input_record, indent=2) + '\n', encoding='utf-8')
print('Manifest rows:', len(rows), '| classes:', len(classes), '| ZIP members:', len(archive_members))
print('All fixed test members are present; per-image payload SHA checks will run during inference.')


In [ ]:
# CUDA activation preflight: run 16 held-out items through the actual ONNX model,
# profile node assignments, and stop unless CUDAExecutionProvider executes model nodes.
from collections import Counter

def make_profiled_session(prefix):
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.profile_file_prefix = str(RUN_DIR / prefix)
    session = ort.InferenceSession(
        str(MODEL),
        sess_options=options,
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
    )
    return session

def profile_provider_counts(profile_path):
    events = json.loads(Path(profile_path).read_text(encoding='utf-8'))
    counts = Counter()
    for event in events:
        if event.get('cat') == 'Node':
            provider = event.get('args', {}).get('provider')
            if provider:
                counts[provider] += 1
    return dict(counts), events

def load_manifest_batch(zip_file, batch_rows):
    tensors = []
    labels = []
    for row in batch_rows:
        raw = zip_file.read(row.member)
        actual_sha = hashlib.sha256(raw).hexdigest()
        if actual_sha.lower() != str(row.sha256).lower():
            raise ValueError(f'Image hash mismatch: {row.member}')
        with Image.open(BytesIO(raw)) as image:
            tensor = preprocess_image(image)[0]
        if tensor.shape != (3, 224, 224) or tensor.dtype != np.float32:
            raise ValueError(f'Unexpected production-preprocessor tensor for {row.member}: {tensor.shape}, {tensor.dtype}')
        tensors.append(tensor)
        labels.append(classes.index(row.class_name))
    return np.stack(tensors).astype(np.float32), np.asarray(labels, dtype=np.int64)

preflight = {
    'status':'started',
    'python_version':sys.version,
    'platform':platform.platform(),
    'pytorch_version':torch.__version__,
    'torch_cuda_build':torch.version.cuda,
    'torch_cudnn_version':torch.backends.cudnn.version(),
    'torch_cuda_available':bool(torch.cuda.is_available()),
    'cuda_device':torch.cuda.get_device_name(0),
    'onnxruntime_version':ort.__version__,
    'available_providers':ort.get_available_providers(),
}
try:
    smoke_session = make_profiled_session('onnx_cuda_smoke_profile')
    preflight['registered_session_providers'] = smoke_session.get_providers()
    if not smoke_session.get_providers() or smoke_session.get_providers()[0] != 'CUDAExecutionProvider':
        raise RuntimeError(f'CUDA provider is not first in session priority: {smoke_session.get_providers()}')
    smoke_rows = list(rows.head(16).itertuples(index=False))
    smoke_batch, _ = load_manifest_batch(archive, smoke_rows)
    smoke_output = smoke_session.run(None, {smoke_session.get_inputs()[0].name:smoke_batch})[0]
    preflight['smoke_output_shape'] = list(smoke_output.shape)
    if smoke_output.shape != (len(smoke_rows), 38):
        raise RuntimeError(f'Unexpected smoke output shape: {smoke_output.shape}')
    preflight['smoke_profile_path'] = smoke_session.end_profiling()
    counts, _ = profile_provider_counts(preflight['smoke_profile_path'])
    preflight['profiled_node_provider_counts'] = counts
    preflight['cuda_node_execution_confirmed'] = counts.get('CUDAExecutionProvider', 0) > 0
    preflight['status'] = 'passed' if preflight['cuda_node_execution_confirmed'] else 'failed_cuda_not_confirmed'
except Exception as error:
    preflight['status'] = 'failed'
    preflight['error_type'] = type(error).__name__
    preflight['error'] = str(error)
    if 'smoke_session' in globals():
        try:
            profile = smoke_session.end_profiling()
            preflight['smoke_profile_path'] = profile
            counts, _ = profile_provider_counts(profile)
            preflight['profiled_node_provider_counts'] = counts
            preflight['cuda_node_execution_confirmed'] = counts.get('CUDAExecutionProvider', 0) > 0
        except Exception as profile_error:
            preflight['profile_error'] = str(profile_error)
(RUN_DIR / 'runtime_preflight.json').write_text(json.dumps(preflight, indent=2) + '\n', encoding='utf-8')
print(json.dumps(preflight, indent=2))
if not preflight.get('cuda_node_execution_confirmed', False):
    raise RuntimeError('CUDAExecutionProvider was not confirmed executing model nodes. Full evaluation stopped; do not report GPU success.')


In [ ]:
# Full evaluation on the unchanged fixed test manifest only.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

BATCH_SIZE = 128
session_options = ort.SessionOptions()
session_options.enable_profiling = True
session_options.profile_file_prefix = str(RUN_DIR / 'onnxruntime_evaluation_profile')
session = ort.InferenceSession(
    str(MODEL),
    sess_options=session_options,
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)
assert session.get_inputs()[0].shape[1:] == [3, 224, 224]
assert session.get_outputs()[0].shape[-1] == 38
input_name = session.get_inputs()[0].name

y_true, y_pred, y_top3 = [], [], []
prediction_rows = []
start_time = time.perf_counter()
for start in range(0, len(rows), BATCH_SIZE):
    batch_rows = list(rows.iloc[start:start+BATCH_SIZE].itertuples(index=False))
    image_batch, labels = load_manifest_batch(archive, batch_rows)
    logits = np.asarray(session.run(None, {input_name:image_batch})[0], dtype=np.float32)
    if logits.shape != (len(batch_rows), 38) or not np.isfinite(logits).all():
        raise RuntimeError(f'Unexpected or non-finite ONNX output at row {start}: {logits.shape}')
    shifted = logits - logits.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    predictions = probabilities.argmax(axis=1)
    top3 = np.argsort(probabilities, axis=1)[:, -3:][:, ::-1]
    y_true.extend(labels.tolist())
    y_pred.extend(predictions.tolist())
    y_top3.extend(top3.tolist())
    for j, row in enumerate(batch_rows):
        prediction_rows.append({
            'member':row.member,
            'sha256':row.sha256,
            'split':row.split,
            'true_index':int(labels[j]),
            'true_class':classes[int(labels[j])],
            'predicted_index':int(predictions[j]),
            'predicted_class':classes[int(predictions[j])],
            'raw_softmax_score_uncalibrated':float(probabilities[j, predictions[j]]),
            'top3_classes':'|'.join(classes[int(i)] for i in top3[j]),
            'top3_indices':'|'.join(str(int(i)) for i in top3[j]),
            'correct_top1':bool(predictions[j] == labels[j]),
            'correct_top3':bool(labels[j] in top3[j]),
        })

evaluation_seconds = time.perf_counter() - start_time
assert len(y_true) == len(rows) == 8154
assert len(prediction_rows) == 8154
report = classification_report(y_true, y_pred, labels=list(range(38)), target_names=classes, output_dict=True, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=list(range(38)))
profile_path = Path(session.end_profiling())
provider_counts, profile_events = profile_provider_counts(profile_path)
final_profile_path = RUN_DIR / 'onnxruntime_profile.json'
profile_path.replace(final_profile_path)
cuda_confirmed_full_run = provider_counts.get('CUDAExecutionProvider', 0) > 0
summary = {
    'run_id':RUN_ID,
    'status':'completed_with_cuda' if cuda_confirmed_full_run else 'completed_cuda_not_confirmed_do_not_claim_gpu',
    'input_hashes_file':'input_hashes.json',
    'runtime_preflight_file':'runtime_preflight.json',
    'images':len(y_true),
    'classes':len(classes),
    'accuracy_top1':float(accuracy_score(y_true, y_pred)),
    'accuracy_top3':float(np.mean([label in pred for label, pred in zip(y_true, y_top3)])),
    'macro_precision':report['macro avg']['precision'],
    'macro_recall':report['macro avg']['recall'],
    'macro_f1':report['macro avg']['f1-score'],
    'weighted_precision':report['weighted avg']['precision'],
    'weighted_recall':report['weighted avg']['recall'],
    'weighted_f1':report['weighted avg']['f1-score'],
    'per_class_metrics':'per_class_metrics.csv',
    'confusion_matrix':'confusion_matrix.csv',
    'providers_available':ort.get_available_providers(),
    'providers_registered':session.get_providers(),
    'profiled_node_provider_counts':provider_counts,
    'onnxruntime_profile':'onnxruntime_profile.json',
    'cuda_execution_confirmed_for_full_run':bool(cuda_confirmed_full_run),
    'batch_size':BATCH_SIZE,
    'evaluation_seconds':evaluation_seconds,
    'image_payload_sha256_verified':len(y_true),
    'test_manifest':'backend/ml/experiments/stage3_preflight/plantvillage_test.csv',
    'test_independence_from_original_training':'unknown; original training image membership unavailable',
    'interpretation':'PlantVillage held-out result only; not independent validation; raw softmax scores are uncalibrated',
}

pd.DataFrame(prediction_rows).to_csv(RUN_DIR / 'predictions.csv', index=False)
pd.DataFrame([{'class_name':name, **report[name]} for name in classes]).to_csv(RUN_DIR / 'per_class_metrics.csv', index=False)
pd.DataFrame(cm, index=classes, columns=classes).to_csv(RUN_DIR / 'confusion_matrix.csv')
fig, ax = plt.subplots(figsize=(18, 16))
image = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(38), classes, rotation=90, fontsize=6)
ax.set_yticks(range(38), classes, fontsize=6)
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_title('PlantVillage fixed test manifest ? 38-class confusion matrix')
fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig(RUN_DIR / 'confusion_matrix.png', dpi=180)
plt.close(fig)
(RUN_DIR / 'evaluation.json').write_text(json.dumps({'summary':summary,'classification_report':report,'class_distribution':class_counts}, indent=2) + '\n', encoding='utf-8')
print(json.dumps(summary, indent=2))
print('Stage 4 artifacts saved to:', RUN_DIR)


## Interpretation

Review `runtime_preflight.json`, `evaluation.json`, and the full `onnxruntime_profile.json` before describing the run. The CUDA claim is supported only if the full-run profile contains node events assigned to `CUDAExecutionProvider`; if it also contains CPU nodes, report the run as hybrid. If the preflight fails, no full metric run is performed. Do not copy outputs into Stage 3 or overwrite the PlantDoc evaluation.

Even a reproduced 100% score would not resolve the original training overlap or establish independent generalization. Keep PlantDoc metrics separate.
